# Lab 3 (completed) — A Conditional Generative Model for Images

Classifier-free guidance, a Diffusion Transformer, a VAE, and latent diffusion. The full MNIST training run lives in `scripts/run_lab3_mnist.py`; here we show the 2-D CFG sanity check and the model wiring.

In [1]:
import math, torch
from flow_matching_labs.distributions import GMM
from flow_matching_labs.paths import GaussianConditionalProbabilityPath, LinearAlpha, LinearBeta
from flow_matching_labs.cfg import MLPConditionalVectorField, CFGTrainer, CFGVectorFieldODE
from flow_matching_labs.core import EulerSimulator
torch.manual_seed(0)

## Q2 Classifier-free guidance on a 3-mode GMM

In [2]:
angles = [0, 2*math.pi/3, 4*math.pi/3]
means = 2 * torch.tensor([[math.cos(a), math.sin(a)] for a in angles])
gmm = GMM(means, torch.tensor([0.2, 0.2, 0.2]), torch.tensor([1/3, 1/3, 1/3]))
path = GaussianConditionalProbabilityPath(gmm, LinearAlpha(), LinearBeta(), p_simple_shape=[2])
vf = MLPConditionalVectorField(dim=2, hidden_dim=256, class_dim=2, num_classes=3)
CFGTrainer(path=path, eta=0.25, null_label=3).train(
    model=vf, num_steps=2000, lr=1e-3, warmup_steps=200, batch_size=250, progress=False)

ode = CFGVectorFieldODE(vf, guidance_scale=1.0, null_label=3)
sim = EulerSimulator(ode)
for mode in range(3):
    y = torch.full((300,), mode).long()
    xs = sim.simulate(path.p_simple.sample(300), torch.linspace(0,1,100).expand(300,-1), y=y)
    print(f"class {mode}: sample mean {xs.mean(0).tolist()} (target {means[mode].tolist()})")

class 0: sample mean [2.0312936305999756, 0.07960420101881027] (target [2.0, 0.0])


class 1: sample mean [-0.9852368235588074, 1.7773305177688599] (target [-1.0, 1.7320507764816284])


class 2: sample mean [-0.9850954413414001, -1.7227681875228882] (target [-1.0, -1.7320507764816284])


## Q3–Q4 Diffusion Transformer + VAE wiring

In [3]:
from flow_matching_labs.dit import DiffusionTransformerFlowModel
from flow_matching_labs.vae import VAE
model = DiffusionTransformerFlowModel(img_size=32, patch_size=4, num_layers=4, c=1, dim=96, heads=4)
print("DiT output:", tuple(model(torch.randn(2,1,32,32), torch.rand(2), torch.zeros(2).long()).shape))
vae = VAE(data_channels=1, hidden_channels=[16,32,64,128], beta=10.0)
zmean, _ = vae.encode(torch.randn(2,1,32,32))
print("VAE latent:", tuple(zmean.shape))  # (2, 128, 4, 4)

DiT output: (2, 1, 32, 32)


VAE latent: (2, 128, 4, 4)


Run `python scripts/run_lab3_mnist.py` to train the DiT on MNIST and generate the labelled digit grids saved under `results/lab3/`.